In [1]:
# ==========================================
# hardware_synthetic_generator.py
# GENERATES HARDWARE-MATCHED DATA (NO WNTR NEEDED)
# ==========================================

import pandas as pd
import numpy as np
import os

# ==========================================
# HARDWARE PROTOTYPE SPECIFICATIONS
# ==========================================

OUT_DIR = "Hardware_Matched_Training_Data"
os.makedirs(OUT_DIR, exist_ok=True)

# Hardware parameters (from your sensor data)
BASE_FLOW_LPM = 2.7
INLET_PRESSURE_SPU = 2.8
OUTLET_PRESSURE_SPU = -0.93

# Pipe specifications
PIPE_DIAMETER = 0.020  # 20mm
PIPE_LENGTH = 1.6      # 160cm total loop
PIPE_ROUGHNESS = 90

# Leak coefficients (controls leak magnitude)
LEAK_COEFFICIENTS = [
    0.0,           # No leak
    1.5e-6,        # Very small leak
    4.0e-6,        # Small leak
    7.8e-6,        # Medium leak
    1.5e-5,        # Larger leak
]

# Leak timing patterns (in seconds)
LEAK_PATTERNS = [
    (0, 0, "no_leak"),
    (50, 100, "early_short"),
    (50, 200, "early_long"),
    (100, 150, "mid_short"),
    (100, 250, "mid_long"),
    (150, 250, "late_long"),
    (30, 280, "sudden_full"),
    (120, 140, "spike"),
]

# Generate scenarios
SCENARIOS = []
scenario_id = 0

for leak_k in LEAK_COEFFICIENTS:
    if leak_k == 0:
        for pattern_start, pattern_end, pattern_name in LEAK_PATTERNS[:1]:
            for noise_level in [0.05, 0.10, 0.15]:
                scenario_id += 1
                SCENARIOS.append({
                    "id": scenario_id,
                    "name": f"no_leak_{noise_level:.2f}_noise_{scenario_id:03d}.csv",
                    "k": leak_k,
                    "start": pattern_start,
                    "end": pattern_end,
                    "pattern": pattern_name,
                    "noise_level": noise_level,
                    "duration_seconds": 300,
                })
    else:
        for pattern_start, pattern_end, pattern_name in LEAK_PATTERNS[1:]:
            for noise_level in [0.05, 0.10, 0.15]:
                scenario_id += 1
                leak_severity = "micro" if leak_k <= 0.2e-5 else "small" if leak_k <= 0.4e-5 else "medium" if leak_k <= 0.6e-5 else "large"
                SCENARIOS.append({
                    "id": scenario_id,
                    "name": f"leak_{leak_severity}_k{leak_k:.1e}_{pattern_name}_{noise_level:.2f}noise_{scenario_id:03d}.csv",
                    "k": leak_k,
                    "start": pattern_start,
                    "end": pattern_end,
                    "pattern": pattern_name,
                    "noise_level": noise_level,
                    "duration_seconds": 300,
                })

print(f"Generated {len(SCENARIOS)} scenario configurations:")
print(f"  No-leak scenarios: {sum(1 for s in SCENARIOS if s['k'] == 0)}")
print(f"  Leak scenarios: {sum(1 for s in SCENARIOS if s['k'] > 0)}")


def generate_synthetic_scenario(k, start_sec, end_sec, noise_level, duration_sec=300):
    n_timesteps = duration_sec
    times = np.arange(n_timesteps)
    demand = np.random.normal(1.0, 0.02, n_timesteps) # Highly stable pump
    
    # Baseline State
    f1_base, f2_base = 2.80 * demand, 2.75 * demand
    p1_base, p2_base = 2.40 * demand, -0.93 * demand
    
    leak_active = np.zeros(n_timesteps, dtype=int)
    if k > 0: leak_active[start_sec:end_sec] = 1
    severity = k / 7.8e-6

    # Real-world deltas from Export 8
    f1 = f1_base + (0.28 * severity * leak_active)
    f2 = f2_base - (0.32 * severity * leak_active)
    p1 = p1_base - (1.60 * severity * leak_active)
    p2 = p2_base - (2.00 * severity * leak_active)

    # ADD CLEAN HARDWARE NOISE (Std Dev ~3.2)
    # Even in "low noise," we match the hardware's 250-sample jitter
    f1 += np.random.normal(0, 0.10, n_timesteps)
    f2 += np.random.normal(0, 0.08, n_timesteps)
    p1 += np.random.normal(0, 3.2 * noise_level * 10, n_timesteps) # Scaled to match 3.26 Std Dev
    p2 += np.random.normal(0, 3.4 * noise_level * 10, n_timesteps)

    return times, p1, f1, f2, p2, leak_active
    
def run_generation():
    """Generate all training scenarios."""
    print("\n" + "=" * 80)
    print("GENERATING HARDWARE-MATCHED TRAINING SCENARIOS (SYNTHETIC)")
    print("=" * 80)
    
    successful = 0
    failed = 0
    
    for scenario in SCENARIOS:
        try:
            print(f"[{scenario['id']:3d}/{len(SCENARIOS)}] {scenario['name']}", end=" ")
            
            # Generate synthetic data
            times, p1, f1, f2, p2, labels = generate_synthetic_scenario(
                k=scenario['k'],
                start_sec=scenario['start'],
                end_sec=scenario['end'],
                noise_level=scenario['noise_level'],
                duration_sec=scenario['duration_seconds']
            )
            
            # Build DataFrame
            df = pd.DataFrame({
                'Time_seconds': times,
                'Pressure_1_SPU': p1,
                'Flow_1_LPM': f1,
                'Flow_2_LPM': f2,
                'Pressure_2_SPU': p2,
                'Label': labels.astype(int)
            })
            
            # ========== ROUND TO MATCH HARDWARE PRECISION ==========
            df['Time_seconds'] = df['Time_seconds'].astype(int)     # Integer
            df['Pressure_1_SPU'] = df['Pressure_1_SPU'].round(2)    # 2 decimals (matches hardware 2.80)
            df['Flow_1_LPM'] = df['Flow_1_LPM'].round(2)            # 2 decimals
            df['Flow_2_LPM'] = df['Flow_2_LPM'].round(2)            # 2 decimals
            df['Pressure_2_SPU'] = df['Pressure_2_SPU'].round(3)    # 3 decimals (matches hardware 0.370)
            df['Label'] = df['Label'].astype(int)                   # Integer
            
            # Save to CSV (no header, matching your hardware format exactly)
            output_path = os.path.join(OUT_DIR, scenario['name'])
            
            # Write with proper formatting
            with open(output_path, 'w') as f:
                for idx, row in df.iterrows():
                    f.write(f"{int(row['Time_seconds'])},{row['Pressure_1_SPU']:.2f},"
                           f"{row['Flow_1_LPM']:.2f},{row['Flow_2_LPM']:.2f},"
                           f"{row['Pressure_2_SPU']:.3f},{int(row['Label'])}\n")
            
            leak_rows = df['Label'].sum()
            leak_pct = 100 * leak_rows / len(df)
            
            print(f"✓ ({leak_pct:.0f}% leak)")
            
            successful += 1
            
        except Exception as e:
            failed += 1
            print(f"✗ Error: {e}")
    
    print("\n" + "=" * 80)
    print(f"GENERATION COMPLETE: {successful} successful, {failed} failed")
    print(f"Total rows generated: {successful * 300}")
    print(f"Output location: {OUT_DIR}/")
    print("=" * 80)

if __name__ == "__main__":
    run_generation()

Generated 87 scenario configurations:
  No-leak scenarios: 3
  Leak scenarios: 84

GENERATING HARDWARE-MATCHED TRAINING SCENARIOS (SYNTHETIC)
[  1/87] no_leak_0.05_noise_001.csv ✓ (0% leak)
[  2/87] no_leak_0.10_noise_002.csv ✓ (0% leak)
[  3/87] no_leak_0.15_noise_003.csv ✓ (0% leak)
[  4/87] leak_micro_k1.5e-06_early_short_0.05noise_004.csv ✓ (17% leak)
[  5/87] leak_micro_k1.5e-06_early_short_0.10noise_005.csv ✓ (17% leak)
[  6/87] leak_micro_k1.5e-06_early_short_0.15noise_006.csv ✓ (17% leak)
[  7/87] leak_micro_k1.5e-06_early_long_0.05noise_007.csv ✓ (50% leak)
[  8/87] leak_micro_k1.5e-06_early_long_0.10noise_008.csv ✓ (50% leak)
[  9/87] leak_micro_k1.5e-06_early_long_0.15noise_009.csv ✓ (50% leak)
[ 10/87] leak_micro_k1.5e-06_mid_short_0.05noise_010.csv ✓ (17% leak)
[ 11/87] leak_micro_k1.5e-06_mid_short_0.10noise_011.csv ✓ (17% leak)
[ 12/87] leak_micro_k1.5e-06_mid_short_0.15noise_012.csv ✓ (17% leak)
[ 13/87] leak_micro_k1.5e-06_mid_long_0.05noise_013.csv ✓ (50% leak)
[ 14/8

In [2]:
import shutil

# This zips the folder into a file named 'training_data.zip'
shutil.make_archive('training_data', 'zip', 'Hardware_Matched_Training_Data')

print("Zip file created successfully!")

Zip file created successfully!
